In [2]:
import sys
import os
import datetime
import argparse
import wandb
from functools import partial
from collections import deque
from types import NoneType
import json
import time
import cupy as cp
import numpy as np
xp = cp
import torch
from layers import Softmax
from model import GoePT
from dataset import Dataset

/home/david/miniconda3/envs/ap/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from torch.nn import functional as F
logits = torch.tensor([[0.3361, 0.0390]],requires_grad=True)
target = torch.tensor([0])
loss=F.cross_entropy(logits,target)
loss.backward()

In [4]:
print(loss)

print(logits.grad)
sm = F.softmax(logits,1)
print(sm-torch.tensor([[1,0]]))

tensor(0.5556, grad_fn=<NllLossBackward0>)
tensor([[-0.4263,  0.4263]])
tensor([[-0.4263,  0.4263]], grad_fn=<SubBackward0>)


In [5]:
one_hot_lookup = xp.eye(2) # 2 = n_genres
def compute_gradient(target, prediction, one_hot_lookup):

    target = xp.stack([one_hot_lookup[token] for token in target]).reshape(prediction.shape)

    grad = prediction - target
    # grad = grad/np.prod(target.shape[:-1])
    return grad, target
with open(os.path.join("../../checkpoints/", 'test2.json'), mode='r', encoding='utf-8') as in_file:
    state_dict = json.load(in_file)


In [8]:
model = GoePT.from_state_dict(state_dict)
x = cp.array([  3, 153, 381,  77, 150, 153, 435,  38, 150, 159, 482, 256, 150, 151,
        482, 262, 142, 151, 482, 269, 142, 151, 217, 482, 262, 142, 151, 462,
         62, 144, 151, 219, 365,  77, 150, 152, 381,  69, 144, 151, 381,  74,
        146, 151, 381,  77, 145, 151, 361,  65, 150, 152, 361,  69, 150, 152,
        361,  74, 148, 152, 482, 266, 142, 151]).reshape((1,-1))
y = cp.array([[0]])

logits, loss = model.forward(x,y,train=False)
print(loss.item())
grad,_target = compute_gradient(y,logits,one_hot_lookup)
model.backward(grad)


0.5585903802456618


In [7]:
def pn(x):
    print(x.mean().item(),x.var().item(),x.min().item(),x.max().item())
x  = model.lm_head.grad_weight
pn(x)
x = model.transformer["ln_f"].grad_weight
pn(x)
print("Blocks")
for block in reversed(model.transformer["h"]):
    x = block.attn.c_attn.grad_weight
    pn(x)

0.0 0.18041681534077758 -0.9584999890044308 0.9584999890044308
-0.0006196615516659887 4.682571320758548e-05 -0.03347617583259155 0.017220338915145596
Blocks
-8.017197558930855e-09 1.1005401111963994e-07 -0.005113540978310897 0.005070458731479179
1.0771821296290909e-10 7.614500532719444e-08 -0.003223005528441533 0.003677211774016535
-5.593095809579597e-09 1.0757534463558839e-07 -0.0041296427829052565 0.004749663735743424
4.9503924734984805e-09 1.6256201276951364e-07 -0.004720771422134034 0.004355162074532432
7.424988172152099e-09 5.712938691274546e-07 -0.0052147798704718406 0.005260924497853661
-1.3591581275688507e-07 3.450611265558919e-05 -0.09384326602815166 0.08792894410140567
